# IR Assignment 7: Web Crawler for Product Information

In [1]:
%pip install -q requests beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Assignment 7: Web Crawler for Product Information
# Website: Books to Scrape
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# ------------------------------------------------------------------
# STEP 1: Set up the website and request headers
# ------------------------------------------------------------------
BASE_URL = "https://books.toscrape.com/"
START_URL = BASE_URL
headers = {
    "User-Agent": "Educational Product Crawler/1.0"
}

products = []
visited_pages = set()

# Limit the number of pages for this demonstration
MAX_PAGES = 5

# ------------------------------------------------------------------
# STEP 2: Crawl product listing pages
# ------------------------------------------------------------------
current_url = START_URL
with requests.Session() as session:
    session.headers.update(headers)
    for page_number in range(MAX_PAGES):
        if not current_url or current_url in visited_pages:
            break

        print(f"\nCrawling page: {current_url}")
        visited_pages.add(current_url)

        try:
            response = session.get(
                current_url,
                timeout=20
            )
            response.raise_for_status()
        except requests.RequestException as error:
            print("Error downloading page:", error)
            break

        # Parse the downloaded HTML
        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # Find all product cards
        product_cards = soup.select("article.product_pod")
        if not product_cards:
            print("No products found on this page.")
            break

        # ----------------------------------------------------------
        # STEP 3: Extract product information
        # ----------------------------------------------------------
        for card in product_cards:
            # Product title and link
            title_tag = card.select_one("h3 a")
            if title_tag is None:
                continue

            product_name = title_tag.get("title", "").strip()
            product_link = urljoin(
                current_url,
                title_tag.get("href", "")
            )

            # Product price
            price_tag = card.select_one(".price_color")
            product_price = (
                price_tag.get_text(strip=True)
                if price_tag else "Not available"
            )

            # Product availability
            availability_tag = card.select_one(
                ".availability"
            )
            availability = (
                availability_tag.get_text(" ", strip=True)
                if availability_tag
                else "Unknown"
            )

            # Store product details
            products.append({
                "Product Name": product_name,
                "Price": product_price,
                "Availability": availability,
                "Product Link": product_link
            })

        print(f"Products collected: {len(products)}")

        # ----------------------------------------------------------
        # STEP 4: Find the next listing page
        # ----------------------------------------------------------
        next_link = soup.select_one("li.next a")
        if next_link:
            current_url = urljoin(
                current_url,
                next_link.get("href", "")
            )
        else:
            current_url = None

        # Be considerate to the website
        if current_url:
            time.sleep(1)

# ------------------------------------------------------------------
# STEP 5: Save results to CSV
# ------------------------------------------------------------------
if products:
    df = pd.DataFrame(products)
    # Remove duplicate product links, if any
    df = df.drop_duplicates(subset=["Product Link"])
    df.to_csv(
        "products.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print("\nCrawling completed successfully.")
    print("Total unique products:", len(df))
    print("Saved results to products.csv")
    print("\nFirst five extracted products:")
    print(df.head().to_string(index=False))
else:
    print("\nNo product information was collected.")

# ------------------------------------------------------------------
# STEP 6: Conclusion
# ------------------------------------------------------------------
print("\nConclusion:")
print("Product information and links were extracted.")
print("The results were stored in a CSV file.")


Crawling page: https://books.toscrape.com/
Products collected: 20

Crawling page: https://books.toscrape.com/catalogue/page-2.html
Products collected: 40

Crawling page: https://books.toscrape.com/catalogue/page-3.html
Products collected: 60

Crawling page: https://books.toscrape.com/catalogue/page-4.html
Products collected: 80

Crawling page: https://books.toscrape.com/catalogue/page-5.html
Products collected: 100

Crawling completed successfully.
Total unique products: 100
Saved results to products.csv

First five extracted products:
                         Product Name   Price Availability                                                                             Product Link
                 A Light in the Attic Â£51.77     In stock                https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
                   Tipping the Velvet Â£53.74     In stock                   https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html
             